# Configurations

In [ ]:
BOOTSTRAP_SERVER = "10.67.22.90:19092" # Kafka port 19092 is dedicated to VMs in CloudVeneto

TOPIC_STREAM = "topic_stream"
TOPIC_RESULTS = "topic_results"

SCHEDULER_IP = "10.67.22.90"
WORKER_IPS = ["10.67.22.58", "10.67.22.244", "10.67.22.210"]
N_WORKERS = len(WORKER_IPS)

WORKER_PUBLISH_INTERVAL_SEC = 5.0

# Topics setup

In [ ]:
from kafka.admin import KafkaAdminClient
from kafka.errors import UnknownTopicOrPartitionError

print("Creating topics...")
# All settings documented in https://kafka-python.readthedocs.io/en/master/apidoc/KafkaAdminClient.html
kafka_admin = KafkaAdminClient(
    bootstrap_servers=BOOTSTRAP_SERVER,
    client_id = "kafka_admin_client",
    security_protocol = "PLAINTEXT"
)

# Delete old topics if they exist
try:
    kafka_admin.delete_topics([TOPIC_STREAM, TOPIC_RESULTS])
except UnknownTopicOrPartitionError:
    pass

# All settings documented in https://kafka.apache.org/42/configuration/topic-configs/
kafka_admin.create_topics({
    TOPIC_STREAM: {
        "num_partitions": N_WORKERS,
        "replication_factor": 1,
        "topic_configs": {
            "message.timestamp.type": "CreateTime", # use producer timestamp instead of log append timestamp
        },
    },
    TOPIC_RESULTS: {
        "num_partitions": 1,
        "replication_factor": 1,
        "topic_configs": {
            "message.timestamp.type": "CreateTime",
        },
    }
})

print("Waiting for topics...")
# Wait until all topics are ready to use
kafka_admin.wait_for_topics([TOPIC_STREAM, TOPIC_RESULTS])
print("Topics ready: ", kafka_admin.list_topics())

 # Close the KafkaAdminClient connection to the Kafka broker
kafka_admin.close()

# Dask cluster setup

Dask needs to be installed in all nodes

Documentation: https://docs.dask.org/en/stable/deploying-ssh.html and https://docs.dask.org/en/stable/_modules/distributed/client.html

In [ ]:
from dask.distributed import Client, SSHCluster

# Disable Dask info logs, keep only warnings and errors
import logging
logging.getLogger("distributed.deploy.ssh").setLevel(logging.WARN)

print("Creating Dask SSH cluster...")
dask_cluster = SSHCluster(
    [SCHEDULER_IP] + WORKER_IPS,
    connect_options = {"known_hosts": None},                              # Disable server host key validation
    worker_options = {"n_workers": 1, "nthreads": 1},                     # Options to pass on to workers
    scheduler_options={"port": 8786, "dashboard_address": "0.0.0.0:8797"} # Options to pass on to scheduler
)

dask_client = Client(dask_cluster, name="dask_client")
dask_client

# Workers code

In [ ]:
class WorkerProcessor:
    """
    One instance of this class runs as a Dask actor on a single worker. See https://distributed.dask.org/en/stable/actors.html
    The actor lifetime is independent of individual streams: BEGIN_STREAM -> process batches -> END_STREAM -> wait for next stream
    """
    
    def process_batch(self, bytestring):
        np = self.np

        BYTES_PER_SCAN = 8 * self.N_SAMPLES_PER_SCAN
    
        batch_size = len(bytestring)
        assert batch_size % BYTES_PER_SCAN == 0
        n_scans = batch_size // BYTES_PER_SCAN
    
        # Split raw bytes into I and Q float32 arrays
        i_data = np.frombuffer(bytestring[:batch_size//2], dtype="<f4") # < = little-endian, f4 = float32
        q_data = np.frombuffer(bytestring[batch_size//2:], dtype="<f4")
        complex_signal = np.empty(i_data.shape, dtype=np.complex64) # this way avoids intermediate allocation
        complex_signal.real = i_data
        complex_signal.imag = q_data
    
        # Shape the signal into a matrix to perform vectorized FFT row-wise
        matrix = complex_signal.reshape((n_scans, self.N_SAMPLES_PER_SCAN))
        # Compute frequency spectrum
        spectra = np.fft.fft(matrix, axis=1)
        power_spectra = np.abs(spectra) ** 2
    
        # Compute mean and second moment of power across scans
        power_means = np.mean(power_spectra, axis=0)
        power_M2s = np.sum((power_spectra - power_means) ** 2, axis=0)
    
        return n_scans, power_means, power_M2s
    

    def __init__(self, worker_id, bootstrap_servers, input_topic, output_topic, publish_interval_sec):
        import json
        import time
        import numpy as np
        from kafka import KafkaConsumer, KafkaProducer

        self.worker_id = worker_id
        self.input_topic = input_topic
        self.output_topic = output_topic
        self.publish_interval_sec = publish_interval_sec

        # Setup consumer. This will read data from the DAQ
        self.consumer = KafkaConsumer(
            input_topic,
            bootstrap_servers=bootstrap_servers,
            client_id = f"worker{worker_id}_consumer", # Appears in server-side logs
            group_id = 'dask_processing_group',        # consumer group to join for dynamic partition assignment
            group_instance_id = f"worker{worker_id}_consumer_instance", # Make this consumer a static member of the group (not really necessary)
            value_deserializer = None,                 # Deserialization will be performed manually
            max_partition_fetch_bytes = 32*1024*1024,  # This size must be at least as large as the maximum message size the server allows
            auto_offset_reset='earliest',              # Bring the reading offset to the earliest message
            security_protocol = "PLAINTEXT", 
        )
    
        # Setup producer. This will publish data for the dashboard
        self.producer = KafkaProducer(
            bootstrap_servers=bootstrap_servers,
            client_id = f"worker{worker_id}_producer", # Appears in server-side logs
            value_serializer=lambda v: json.dumps(v).encode('utf-8'),
            compression_type=None,                     # No compression
            enable_idempotence = True,                 # Ensure that exactly one copy of each message is written in the stream
            batch_size = 0,                            # Disable batching, we will do our own
            max_block_ms = 3000,                       # Max block time if buffer is full
            max_request_size = 256*1024,               # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
            max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
            security_protocol = "PLAINTEXT"
        )

        self.N_SAMPLES_PER_SCAN = 2048 # Number of complex samples in each scan
        SAMPLING_FREQ_HZ = 2e6       # ADC readout frequency
        # Compute frequency bins
        self.frequencies = np.fft.fftfreq(self.N_SAMPLES_PER_SCAN, d = 1 / SAMPLING_FREQ_HZ).tolist()

        self.stream_active = False
        self.n_scans_per_batch = None
        self.producer_throughput_MB = None

        self.publish_deadline = None

        # Worker imports need to execute on the worker process
        self.np = np
        self.json = json
        self.time = time


    def __del__(self):
        try:
            self.consumer.close()
            self.producer.close()
        except Exception:
            pass


    def begin_stream(self, header_value, begin_timestamp):
        metadata = self.json.loads(header_value.decode("utf-8"))

        body = {
            "worker_id": self.worker_id,
            "throughput_MB": metadata["throughput_MB"],
            "n_scans_per_batch": metadata["n_scans_per_batch"],
            "producer_begin_ts": begin_timestamp,
            "frequencies": self.frequencies,
        }
        self.producer.send(self.output_topic, value=body, headers=[('BEGIN_STREAM', b"")])
        self.producer.flush()

        self.n_scans_per_batch = metadata["n_scans_per_batch"]
        self.producer_throughput_MB = metadata["throughput_MB"]

        self.reset_accumulators()
        self.stream_active = True

        self.publish_deadline = self.time.monotonic() + self.publish_interval_sec


    def end_stream(self, end_timestamp):
        if self.accumulated_n_scans != 0:
            self.publish()
        body = {
            "worker_id": self.worker_id,
            "producer_end_ts": end_timestamp,
        }
        self.producer.send(self.output_topic, value=body, headers=[('END_STREAM', b"")])
        self.producer.flush()

        self.reset_accumulators()
        self.stream_active = False
        self.n_scans_per_batch = None
        self.producer_throughput_MB = None
        self.publish_deadline = None


    def reset_accumulators(self):
        self.accumulated_means = self.np.array([])
        self.accumulated_M2s = self.np.array([])
        self.accumulated_n_scans = 0

        self.producer_timestamps = []
        self.batch_completion_times = []
        self.processing_times = []


    # Merge batch mean/M2 statistics using Welford's algorithm
    def update_statistics(self, n_scans_in_batch, batch_means, batch_M2s):
        if self.accumulated_n_scans == 0:
            self.accumulated_means = batch_means.copy()
            self.accumulated_M2s = batch_M2s.copy()
            self.accumulated_n_scans = n_scans_in_batch
        else:
            deltas = batch_means - self.accumulated_means
            total_n_scans = self.accumulated_n_scans + n_scans_in_batch
            self.accumulated_means += deltas * n_scans_in_batch / total_n_scans
            self.accumulated_M2s += batch_M2s + deltas**2 * self.accumulated_n_scans * n_scans_in_batch / total_n_scans
            self.accumulated_n_scans = total_n_scans
    

    def publish(self):
        now = self.time.monotonic()
        result = {
            "worker_id": self.worker_id,
            "results": {
               "n_averaged_scans": self.accumulated_n_scans,
                "power_means": self.accumulated_means.tolist(),
                "power_M2s": self.accumulated_M2s.tolist(),
                "producer_timestamps": self.producer_timestamps,                           # Timestamp reported by producer for each batch
                "waiting_times": [now - finish_time for finish_time in self.batch_completion_times], # How long each batch has been held due to publishing interval logic
                "processing_times" : self.processing_times,                                # Time to process each batch
            } 
        }
        self.producer.send(self.output_topic, value=result)
        self.producer.flush()
        self.publish_deadline = self.time.monotonic() + self.publish_interval_sec


    def handle_signal(self, msg):
        if msg.headers is not None:
            for key, value in msg.headers:
                print("Received header with key ", key)
                if key == "END_STREAM":
                    if self.stream_active is True: # For the case n_workers < n_partitions
                        self.end_stream(msg.timestamp/1000)
                    return True
                elif key == "BEGIN_STREAM":
                    if self.stream_active is False:
                        self.begin_stream(value, msg.timestamp/1000)
                    return True
        return False


    # This method is submitted once per actor and only returns if the worker fails or an exception is raised
    def run(self):
        for msg in self.consumer:
            print("Received message")
            if self.handle_signal(msg):
                continue

            if not self.stream_active:
                raise RuntimeError("Received data before BEGIN_STREAM")

            self.producer_timestamps.append(msg.timestamp/1000)
            processing_time_start = self.time.perf_counter()

            n_scans, means, M2s = self.process_batch(msg.value)
            self.update_statistics(n_scans, means, M2s)

            self.processing_times.append(self.time.perf_counter() - processing_time_start)
            self.batch_completion_times.append(self.time.time())

            if self.time.monotonic() > self.publish_deadline:
                self.publish()
                self.reset_accumulators()

# Temp no Dask

In [ ]:
wp = WorkerProcessor(
        # Parameters to constructor
        worker_id = 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,
        publish_interval_sec = WORKER_PUBLISH_INTERVAL_SEC,)
wp.run()

# Start workers

In [ ]:
def report_status(future):
    if future.status == "error":
        print(f"Task {future.key} failed:")
        print(future.exception())
    else:
        print(f"Task {future.key} finished with status {future.status}")

actors = []
for (i, worker_ip) in enumerate(WORKER_IPS):
    actor_future = dask_client.submit(
        WorkerProcessor,
        # Parameters to constructor
        worker_id = i + 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,
        publish_interval_sec = WORKER_PUBLISH_INTERVAL_SEC,

        # Dask parameters
        key = f"worker{i+1}", # Identifier for the task
        workers = worker_ip,  # Restrict execution to the specific worker
        resources = {},       # no resources (eg GPUs) required
        actor=True,           # Actor pattern
    )
    actor = actor_future.result()
    actor.run()
    actors.append(actor)

# Close resources

In [ ]:
dask_client.close()
dask_cluster.close()